In [1]:
import plotly.express as px
import pandas as pd
import os, sys

module_path = os.path.abspath(os.path.join('../../../..')) # or the path to your source code
sys.path.insert(0, module_path)
import util

config = util.summary_settings
input_config = util.input_settings

In [12]:
hh = util.get_hh_data(quantile_groups=['hhincome','emptot_1','hh_1'])

    # join(util.get_parcel_landuse_data(), how="left",left_on='hhno',right_on='parcelid').\
df_hh = hh.\
    join(util.get_parcel_geog(), how="left",left_on='hhparcel',right_on='ParcelID').\
        to_pandas()


Total Households

In [5]:
df = df_hh.groupby('source')['hhexpfac'].sum().reset_index()
df['Total Households'] = df['hhexpfac'].apply(lambda x: f"{x:,.0f}")

df[['source','Total Households']]

,source,Total Households
0,model,"1,736,129"
1,survey,"1,690,793"


- income, hh density, employment density grouped into very low, low, medium, medium-high and high

In [7]:
# Group income, hh density, and employment density into 4 groups
var_group = df_hh.loc[(df_hh['source'] == 'model') & (df_hh['hhincome']>=0),['hhincome','emptot_1','hh_1']].quantile([.125, .25, .50, .75])

var_group

,hhincome,emptot_1,hh_1
0.125,30957.0,0.000000,35.219832
0.250,56275.0,0.399123,86.460702
0.500,106058.0,39.000000,208.434002
0.750,180194.0,222.873746,410.892532


In [8]:
def plot_hh_stat(df:pd.DataFrame, var:str, title_cat:str, wid = 700):
    df_plot = df.groupby(['source',var])['hhexpfac'].sum().reset_index()
    df_plot['percentage'] = df_plot.groupby(['source'], group_keys=False)['hhexpfac'].\
            apply(lambda x: x / float(x.sum()))
    
    df_plot_ct = df.groupby(['source',var])['hhexpfac'].count().reset_index(). \
        rename(columns={'hhexpfac':'sample count'})
    df_plot = df_plot.merge(df_plot_ct, on=['source',var])
    
    fig = px.bar(df_plot.sort_values(by=['source']), x=var, y="percentage", color="source",
                 hover_data=['sample count'],
                 barmode="group",title=title_cat)
    fig.update_layout(height=400, width=wid, font=dict(size=11),
                      yaxis=dict(tickformat=".2%"))
    fig.show()

## demographics

In [13]:
plot_hh_stat(df_hh, 'hhincome_4group', 'household income')

## Home Location

In [14]:
df_hh[df_hh['source']=="model"]['CountyName'].value_counts()

CountyName
King              957460
Pierce            350141
Snohomish         319624
Kitsap            108901
Outside Region         3
Name: count, dtype: int64

In [15]:
plot_hh_stat(df_hh, 'CountyName', 'home county')

In [16]:
plot_hh_stat(df_hh, 'district_name', 'home district', wid=900)

In [17]:
plot_hh_stat(df_hh, 'hh_1_4group', 'home density')